# Derma-AI: entrenamiento y evaluación del modelo CNN (HAM10000)

Cuaderno para **Google Colab**. Entrena un clasificador de 7 clases con **MobileNetV2** (transferencia de aprendizaje) y lo evalúa en un conjunto de prueba independiente.

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

Pasos: 1) GPU y Drive, 2) descargar datos, 3) dividir sin fuga de datos, 4) entrenar, 5) evaluar, 6) guardar.

> Prototipo académico. Las métricas de este cuaderno **no constituyen validación clínica**.

## 1. GPU y Google Drive

In [ ]:
import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))  # debe aparecer una GPU; si sale [], activa la GPU en el entorno de ejecución

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
SALIDA = '/content/drive/MyDrive/derma-ai/modelo_v1'   # aquí quedan el modelo y las métricas
os.makedirs(SALIDA, exist_ok=True)
print('Se guardará en:', SALIDA)

## 2. Descargar HAM10000 (Kaggle)

1. En Kaggle: *Settings → API → Create New Token*. Se descarga `kaggle.json`.
2. Ejecuta la celda y sube ese archivo cuando lo pida.

**Licencia (actividad 1016):** verifica y anota la licencia en la página del conjunto de datos. El conjunto original (Tschandl et al., 2018, Harvard Dataverse) se distribuye bajo CC BY-NC, es decir, uso no comercial, compatible con un proyecto académico. Documenta la fuente y la versión en el informe.

In [ ]:
from google.colab import files
files.upload()   # sube kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham --unzip
!ls /content/ham | head

## 3. Metadatos y división sin fuga de datos

In [ ]:
import pandas as pd, numpy as np, glob, os

DATA = '/content/ham'
meta = pd.read_csv(f'{DATA}/HAM10000_metadata.csv')

# Mapa image_id -> ruta del archivo (busca en todas las subcarpetas)
rutas = {}
for p in glob.glob(f'{DATA}/**/*.jpg', recursive=True):
    rutas[os.path.splitext(os.path.basename(p))[0]] = p
meta['ruta'] = meta['image_id'].map(rutas)
assert meta['ruta'].notna().all(), 'Faltan imágenes: revisa la carpeta DATA'

CLASES = sorted(meta['dx'].unique())          # akiec, bcc, bkl, df, mel, nv, vasc
meta['y'] = meta['dx'].map({c: i for i, c in enumerate(CLASES)})
print(len(meta), 'imágenes,', meta['lesion_id'].nunique(), 'lesiones distintas')
print(meta['dx'].value_counts())

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

# Varias imágenes pertenecen a la misma lesión (lesion_id). Se agrupa por lesion_id para que
# una lesión nunca quede repartida entre entrenamiento y prueba (evita fuga de datos).
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
folds = [idx for _, idx in sgkf.split(meta, meta['y'], groups=meta['lesion_id'])]
idx_test, idx_val = folds[0], folds[1]
idx_train = np.setdiff1d(np.arange(len(meta)), np.concatenate([idx_test, idx_val]))
train, val, test = meta.iloc[idx_train], meta.iloc[idx_val], meta.iloc[idx_test]

# Comprobación: ninguna lesión en dos particiones
assert not set(train.lesion_id) & set(val.lesion_id)
assert not set(train.lesion_id) & set(test.lesion_id)
assert not set(val.lesion_id) & set(test.lesion_id)

for nombre, df in [('train', train), ('val', val), ('test', test)]:
    print(nombre, len(df), (df['dx'].value_counts(normalize=True).round(3)).to_dict())
    df[['image_id', 'lesion_id', 'dx']].to_csv(f'{SALIDA}/split_{nombre}.csv', index=False)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Las clases están muy desbalanceadas (nv domina). Se pondera la pérdida; la raíz cuadrada
# suaviza los pesos para que el entrenamiento sea estable.
pesos = compute_class_weight('balanced', classes=np.arange(len(CLASES)), y=train['y'].values)
class_weight = {i: float(np.sqrt(w)) for i, w in enumerate(pesos)}
print({CLASES[i]: round(w, 2) for i, w in class_weight.items()})

## 4. Entrenamiento

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG, BATCH, AUTOTUNE = 224, 32, tf.data.AUTOTUNE

def cargar(ruta, y):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    img = tf.image.resize(img, (IMG, IMG))
    return tf.cast(tf.round(img), tf.uint8), y          # uint8 para poder usar cache()

aumento = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
])

def hacer_ds(df, entrenar):
    ds = tf.data.Dataset.from_tensor_slices((df['ruta'].values, df['y'].values.astype('int32')))
    ds = ds.map(cargar, num_parallel_calls=AUTOTUNE).cache()
    if entrenar:
        ds = ds.shuffle(2048, seed=42)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    if entrenar:
        ds = ds.map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)  # escala a [-1, 1]
    return ds.prefetch(AUTOTUNE)

ds_train, ds_val, ds_test = hacer_ds(train, True), hacer_ds(val, False), hacer_ds(test, False)

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet')
base.trainable = False

inputs = tf.keras.Input((IMG, IMG, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CLASES), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cb = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]
# Fase 1: solo se entrena la cabeza clasificadora
h1 = model.fit(ds_train, validation_data=ds_val, epochs=10, class_weight=class_weight, callbacks=cb)

In [ ]:
# Fase 2: ajuste fino de las últimas capas con tasa de aprendizaje baja
base.trainable = True
for capa in base.layers[:-40]:
    capa.trainable = False
for capa in base.layers:
    if isinstance(capa, layers.BatchNormalization):
        capa.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
h2 = model.fit(ds_train, validation_data=ds_val, epochs=12, class_weight=class_weight, callbacks=cb)

## 5. Evaluación en el conjunto de prueba

Esta es la única vez que se usa el conjunto de prueba (requisito 8.6). Se reportan métricas **ponderadas** (por tamaño de clase) y **macro** (promedio simple entre clases). El acta no indica cuál promedio aplica a los criterios CE-01 a CE-03: acuérdalo con el equipo y el docente. Como el nevus domina, la exactitud global suele verse mejor que las métricas macro.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt, json, datetime

nombres = [c.upper() for c in CLASES]
y_true = test['y'].values
y_pred = model.predict(ds_test).argmax(axis=1)      # ds_test no se baraja: el orden coincide con `test`

exactitud = accuracy_score(y_true, y_pred)
rep = classification_report(y_true, y_pred, target_names=nombres, output_dict=True, zero_division=0)
print(f'Exactitud: {exactitud:.3f}')
print(classification_report(y_true, y_pred, target_names=nombres, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap='Blues'); ax.set_xticks(range(7)); ax.set_yticks(range(7))
ax.set_xticklabels(nombres, rotation=45); ax.set_yticklabels(nombres)
ax.set_xlabel('Predicha'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusión (prueba)')
for i in range(7):
    for j in range(7):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout(); plt.savefig(f'{SALIDA}/matriz_confusion.png', dpi=150); plt.show()

## 6. Guardar modelo, clases y métricas

In [ ]:
model.save(f'{SALIDA}/dermaai_mobilenetv2_v1.keras')

json.dump(nombres, open(f'{SALIDA}/class_names.json', 'w'), ensure_ascii=False)
json.dump({
    'arquitectura': 'MobileNetV2',
    'tamano_entrada': [IMG, IMG, 3],
    'preprocesamiento': 'tf.keras.applications.mobilenet_v2.preprocess_input (escala los píxeles a [-1, 1])',
    'clases': nombres,
}, open(f'{SALIDA}/model_config.json', 'w'), ensure_ascii=False, indent=2)

metricas = {
    'fecha': datetime.date.today().isoformat(),
    'conjunto': 'HAM10000, partición de prueba agrupada por lesion_id',
    'n_prueba': int(len(test)),
    'exactitud': float(exactitud),
    'precision_macro': rep['macro avg']['precision'], 'sensibilidad_macro': rep['macro avg']['recall'], 'f1_macro': rep['macro avg']['f1-score'],
    'precision_ponderada': rep['weighted avg']['precision'], 'sensibilidad_ponderada': rep['weighted avg']['recall'], 'f1_ponderada': rep['weighted avg']['f1-score'],
    'por_clase': {n: {k: rep[n][k] for k in ('precision', 'recall', 'f1-score', 'support')} for n in nombres},
    'hiperparametros': {'fase1': {'epocas_max': 10, 'lr': 1e-3}, 'fase2': {'epocas_max': 12, 'lr': 1e-5, 'capas_descongeladas': 40}, 'batch': BATCH, 'semilla': 42},
}
json.dump(metricas, open(f'{SALIDA}/metrics.json', 'w'), ensure_ascii=False, indent=2)
print('Guardado en', SALIDA)
print({k: round(v, 3) for k, v in metricas.items() if isinstance(v, float)})

## Qué hacer después
1. Pasa `exactitud`, `sensibilidad` y `f1` (macro y ponderadas) al equipo.
2. Descarga `dermaai_mobilenetv2_v1.keras`, `class_names.json` y `model_config.json` desde Drive y cópialos a `models/` del repositorio.
3. Si las métricas macro quedan por debajo del mínimo (80 % de sensibilidad, 0,75 de F1), anota las causas y las pruebas en el registro de ajustes (actividad 1022) antes de reentrenar.